# VeReMi — Causal Node-State Completion

This notebook performs the **final controlled transformation** before freezing **Graph Dataset V1.0**.

### Problem resolved
The integrity audit identified **1,792 node-snapshot rows** with:
- valid current-snapshot graph topology;
- genuine communication-edge participation;
- no contemporaneous Type-2 local state;
- a valid local state in the immediately preceding snapshot;
- no need for future-state information.

### Final treatment
For those rows only:

1. Carry forward the most recent **state-derived attributes** from the immediately preceding snapshot.
2. Preserve the **current-snapshot topology** exactly as computed.
3. Add:
   - `state_available_current`
   - `state_age_snapshots`
4. Never copy labels, topology, identifiers, graph structure, or future data.

### Input
`GRAPH_FEATURE_ROOT` configured through `src/config.py`

### Output
`GRAPH_DATASET_ROOT` configured through `src/config.py`

The source dataset is never modified.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

from src.config import GRAPH_FEATURE_ROOT, GRAPH_DATASET_ROOT
SOURCE_ROOT = GRAPH_FEATURE_ROOT
SOURCE_ARCH = SOURCE_ROOT / "archives"

OUTPUT_ROOT = GRAPH_DATASET_ROOT
OUTPUT_ARCH = OUTPUT_ROOT / "archives"

OUTPUT_ARCH.mkdir(parents=True, exist_ok=True)

archive_dirs = sorted(
    p for p in SOURCE_ARCH.iterdir()
    if p.is_dir()
)

EXPECTED_ARCHIVES = 225
EXPECTED_COMPLETED_ROWS = 1792

print("Source archives:", len(archive_dirs))
print("Output:", OUTPUT_ROOT)

if len(archive_dirs) != EXPECTED_ARCHIVES:
    print(
        f"WARNING: Expected {EXPECTED_ARCHIVES} archives, "
        f"found {len(archive_dirs)}"
    )


In [ ]:
# ============================================================
# 1. DEFINE EXACT STATE COLUMNS ALLOWED FOR CARRY-FORWARD
# ============================================================

STATE_COLUMNS = [
    "A_flag",
    "time",
    "pos_x",
    "pos_y",
    "pos_z",
    "spd_x",
    "spd_y",
    "spd_z",
    "local_noise_x",
    "local_noise_y",
    "local_noise_z",
    "spd_noise_x",
    "spd_noise_y",
    "spd_noise_z",
    "speed_magnitude",
    "motion_angle",
]

# These are derived from state history. We do not directly copy them.
# They will be recomputed or left according to causal semantics later.
STATE_CHANGE_COLUMNS = [
    "delta_speed_magnitude",
    "snapshot_displacement",
    "delta_motion_angle",
]

TOPOLOGY_COLUMNS = [
    "in_degree",
    "out_degree",
    "degree",
    "degree_centrality",
    "pagerank",
    "clustering_coefficient",
    "component_size",
]

print("Carry-forward state columns:", len(STATE_COLUMNS))
print("Current-topology columns preserved:", len(TOPOLOGY_COLUMNS))


## Important design rule

We do **not** blindly fill every null.

Only rows where the contemporaneous local state is absent are treated. The previous snapshot's physical/state attributes are copied forward. Current topology is preserved.

For transparency:

- `state_available_current = 1` → true local state exists in current snapshot.
- `state_available_current = 0` → state carried from previous snapshot.
- `state_age_snapshots = 0` → current state.
- `state_age_snapshots = 1` → carried one snapshot forward.

The 1,792 previously diagnosed rows are expected to receive age `1`.


In [ ]:
# ============================================================
# 2. CORE COMPLETION FUNCTION
# ============================================================

def complete_node_states(node_df: pd.DataFrame):
    """
    Causally complete missing local-state rows using only snapshot t-1.

    Returns:
      completed_df
      audit_records
    """

    df = node_df.copy()

    if "archive_name" not in df.columns:
        # archive_name may be absent on topology-only rows because it came
        # from the state side of an outer merge.
        df["archive_name"] = pd.NA

    df = df.sort_values(
        ["vehicle_id", "snapshot_id"]
    ).reset_index(drop=True)

    # Mark whether contemporaneous local state was present originally.
    df["state_available_current"] = (
        df["time"].notna()
    ).astype("int8")

    df["state_age_snapshots"] = np.where(
        df["state_available_current"] == 1,
        0,
        np.nan
    )

    audit = []

    # Index by vehicle/snapshot for exact t-1 lookup.
    lookup = {
        (int(row.vehicle_id), int(row.snapshot_id)): idx
        for idx, row in df[["vehicle_id", "snapshot_id"]].iterrows()
        if pd.notna(row.vehicle_id) and pd.notna(row.snapshot_id)
    }

    missing_indices = df.index[
        df["time"].isna()
    ].tolist()

    for idx in missing_indices:

        vehicle_id = int(df.at[idx, "vehicle_id"])
        snapshot_id = int(df.at[idx, "snapshot_id"])

        prev_key = (
            vehicle_id,
            snapshot_id - 1
        )

        prev_idx = lookup.get(prev_key)

        if prev_idx is None:
            audit.append({
                "row_index": idx,
                "vehicle_id": vehicle_id,
                "snapshot_id": snapshot_id,
                "status": "FAIL_NO_PREVIOUS_SNAPSHOT_ROW"
            })
            continue

        # Previous row itself must contain genuine state.
        if pd.isna(df.at[prev_idx, "time"]):
            audit.append({
                "row_index": idx,
                "vehicle_id": vehicle_id,
                "snapshot_id": snapshot_id,
                "status": "FAIL_PREVIOUS_STATE_MISSING"
            })
            continue

        # Carry only approved state columns.
        for col in STATE_COLUMNS:
            if col in df.columns:
                df.at[idx, col] = df.at[prev_idx, col]

        # archive_name is provenance metadata, not a model feature,
        # but it should be populated consistently.
        if pd.isna(df.at[idx, "archive_name"]):
            df.at[idx, "archive_name"] = df.at[prev_idx, "archive_name"]

        df.at[idx, "state_available_current"] = 0
        df.at[idx, "state_age_snapshots"] = 1

        audit.append({
            "row_index": idx,
            "vehicle_id": vehicle_id,
            "snapshot_id": snapshot_id,
            "status": "COMPLETED_FROM_PREVIOUS_SNAPSHOT"
        })

    # Enforce compact integer type after successful completion.
    df["state_age_snapshots"] = pd.to_numeric(
        df["state_age_snapshots"],
        errors="coerce"
    ).astype("Int8")

    return df, pd.DataFrame(audit)


In [ ]:
# ============================================================
# 3. RECOMPUTE CAUSAL STATE-CHANGE FEATURES
# ============================================================

def recompute_state_change_features(df: pd.DataFrame):
    """
    Recompute state-change features after causal completion.

    These are based only on current completed state and t-1 completed state.
    No future information is used.
    """

    x = df.sort_values(
        ["vehicle_id", "snapshot_id"]
    ).copy()

    g = x.groupby(
        "vehicle_id",
        sort=False
    )

    if "speed_magnitude" in x.columns:
        x["delta_speed_magnitude"] = (
            g["speed_magnitude"].diff()
        )

    if {"pos_x", "pos_y"}.issubset(x.columns):
        dx = g["pos_x"].diff()
        dy = g["pos_y"].diff()

        x["snapshot_displacement"] = np.sqrt(
            dx ** 2 + dy ** 2
        )

    if "motion_angle" in x.columns:
        raw = g["motion_angle"].diff()

        x["delta_motion_angle"] = np.arctan2(
            np.sin(raw),
            np.cos(raw)
        )

    return x.sort_values(
        ["snapshot_id", "vehicle_id"]
    ).reset_index(drop=True)


## Three-archive dry run

Run this first. We expect:

- no processing failures;
- current topology unchanged;
- no missing state in rows that were previously anomalous;
- only legitimate first-observation temporal deltas may remain null;
- `state_age_snapshots` should be only 0 or 1.


In [ ]:
# ============================================================
# 4. DRY RUN ON FIRST 3 ARCHIVES
# ============================================================

dry_records = []
dry_failures = []

for i, src_dir in enumerate(
    archive_dirs[:3],
    start=1
):

    print(f"[{i}/3] {src_dir.name}")

    try:

        node_df = pd.read_parquet(
            src_dir / "node_features.parquet"
        )

        original_topology = node_df[
            [
                "snapshot_id",
                "vehicle_id",
                *[
                    c for c in TOPOLOGY_COLUMNS
                    if c in node_df.columns
                ]
            ]
        ].copy()

        before_missing = int(
            node_df["time"].isna().sum()
        )

        completed, audit_df = complete_node_states(
            node_df
        )

        completed = recompute_state_change_features(
            completed
        )

        after_missing = int(
            completed["time"].isna().sum()
        )

        # Verify topology has not changed.
        after_topology = completed[
            original_topology.columns
        ].copy()

        topo_check = original_topology.merge(
            after_topology,
            on=["snapshot_id", "vehicle_id"],
            suffixes=("_before", "_after"),
            validate="one_to_one"
        )

        topo_diff = 0

        for col in TOPOLOGY_COLUMNS:
            if (
                col + "_before" in topo_check.columns
                and
                col + "_after" in topo_check.columns
            ):

                a = topo_check[col + "_before"]
                b = topo_check[col + "_after"]

                topo_diff += int(
                    ~np.isclose(
                        a.to_numpy(dtype=float),
                        b.to_numpy(dtype=float),
                        equal_nan=True
                    )
                ).sum()

        invalid_age = int(
            ~completed[
                "state_age_snapshots"
            ].isin([0, 1]).sum()
            if False else 0
        )

        ages = set(
            completed[
                "state_age_snapshots"
            ].dropna().astype(int).unique()
        )

        dry_records.append({
            "archive_name": src_dir.name,
            "missing_state_before": before_missing,
            "missing_state_after": after_missing,
            "completed_rows": int(
                (
                    audit_df["status"]
                    ==
                    "COMPLETED_FROM_PREVIOUS_SNAPSHOT"
                ).sum()
            ) if not audit_df.empty else 0,
            "failed_completion_rows": int(
                (
                    audit_df["status"]
                    !=
                    "COMPLETED_FROM_PREVIOUS_SNAPSHOT"
                ).sum()
            ) if not audit_df.empty else 0,
            "topology_value_changes": topo_diff,
            "state_age_values": str(sorted(ages))
        })

        print("SUCCESS")

    except Exception as e:

        dry_failures.append({
            "archive_name": src_dir.name,
            "error": repr(e)
        })

        print("FAILED:", repr(e))

dry_summary = pd.DataFrame(
    dry_records
)

dry_failure_df = pd.DataFrame(
    dry_failures
)

display(dry_summary)
display(dry_failure_df)


If the dry run succeeds, execute the full production cell below.


In [ ]:
# ============================================================
# 5. FULL 225-ARCHIVE PRODUCTION
# ============================================================

production_records = []
failure_records = []
completion_audits = []

for i, src_dir in enumerate(
    archive_dirs,
    start=1
):

    print(
        f"[{i}/{len(archive_dirs)}] "
        f"{src_dir.name}"
    )

    try:

        node_df = pd.read_parquet(
            src_dir / "node_features.parquet"
        )

        edge_src = (
            src_dir /
            "edge_features.parquet"
        )

        snap_src = (
            src_dir /
            "snapshot_features.parquet"
        )

        # Capture topology before transformation.
        topology_cols_present = [
            c for c in TOPOLOGY_COLUMNS
            if c in node_df.columns
        ]

        original_topology = node_df[
            [
                "snapshot_id",
                "vehicle_id",
                *topology_cols_present
            ]
        ].copy()

        missing_before = int(
            node_df["time"].isna().sum()
        )

        completed, audit_df = (
            complete_node_states(
                node_df
            )
        )

        completed = (
            recompute_state_change_features(
                completed
            )
        )

        missing_after = int(
            completed["time"].isna().sum()
        )

        if not audit_df.empty:
            audit_df["archive_name"] = src_dir.name
            completion_audits.append(
                audit_df
            )

        failed_rows = (
            int(
                (
                    audit_df["status"]
                    !=
                    "COMPLETED_FROM_PREVIOUS_SNAPSHOT"
                ).sum()
            )
            if not audit_df.empty
            else 0
        )

        completed_rows = (
            int(
                (
                    audit_df["status"]
                    ==
                    "COMPLETED_FROM_PREVIOUS_SNAPSHOT"
                ).sum()
            )
            if not audit_df.empty
            else 0
        )

        if failed_rows != 0:
            raise ValueError(
                f"{failed_rows} rows could not be causally completed"
            )

        if missing_after != 0:
            raise ValueError(
                f"{missing_after} local-state timestamps remain missing"
            )

        # Check state age.
        valid_age = completed[
            "state_age_snapshots"
        ].isin([0, 1])

        if not valid_age.all():
            raise ValueError(
                "Unexpected state_age_snapshots values"
            )

        # Verify current topology is unchanged.
        new_topology = completed[
            original_topology.columns
        ]

        cmp = original_topology.merge(
            new_topology,
            on=[
                "snapshot_id",
                "vehicle_id"
            ],
            suffixes=(
                "_before",
                "_after"
            ),
            validate="one_to_one"
        )

        topology_changes = 0

        for col in topology_cols_present:

            a = cmp[
                col + "_before"
            ].to_numpy(
                dtype=float
            )

            b = cmp[
                col + "_after"
            ].to_numpy(
                dtype=float
            )

            topology_changes += int(
                (
                    ~np.isclose(
                        a,
                        b,
                        equal_nan=True
                    )
                ).sum()
            )

        if topology_changes != 0:
            raise ValueError(
                f"Topology changed in {topology_changes} cells"
            )

        # Output archive.
        out_dir = (
            OUTPUT_ARCH /
            src_dir.name
        )

        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        completed.to_parquet(
            out_dir /
            "node_features.parquet",
            index=False,
            compression="zstd"
        )

        # Edge and snapshot features are unchanged.
        shutil.copy2(
            edge_src,
            out_dir /
            "edge_features.parquet"
        )

        shutil.copy2(
            snap_src,
            out_dir /
            "snapshot_features.parquet"
        )

        production_records.append({
            "archive_name":
                src_dir.name,

            "node_rows":
                len(completed),

            "missing_state_before":
                missing_before,

            "completed_state_rows":
                completed_rows,

            "missing_state_after":
                missing_after,

            "topology_changes":
                topology_changes,

            "state_available_current_0":
                int(
                    (
                        completed[
                            "state_available_current"
                        ] == 0
                    ).sum()
                ),

            "state_available_current_1":
                int(
                    (
                        completed[
                            "state_available_current"
                        ] == 1
                    ).sum()
                )
        })

        print("SUCCESS")

    except Exception as e:

        failure_records.append({
            "archive_name":
                src_dir.name,

            "error":
                repr(e)
        })

        print(
            "FAILED:",
            repr(e)
        )


production_df = pd.DataFrame(
    production_records
)

failure_df = pd.DataFrame(
    failure_records
)

if completion_audits:

    completion_audit_df = pd.concat(
        completion_audits,
        ignore_index=True
    )

else:

    completion_audit_df = pd.DataFrame()


production_df.to_csv(
    OUTPUT_ROOT /
    "01_causal_completion_summary.csv",
    index=False
)

failure_df.to_csv(
    OUTPUT_ROOT /
    "02_causal_completion_failures.csv",
    index=False
)

completion_audit_df.to_csv(
    OUTPUT_ROOT /
    "03_completed_rows_audit.csv",
    index=False
)

print("\n" + "=" * 70)
print("PRODUCTION COMPLETE")
print("=" * 70)

print("Completed archives:", len(production_df))
print("Failed archives   :", len(failure_df))

display(production_df.head())
display(failure_df.head())


In [ ]:
# ============================================================
# 6. FINAL V1.0 VALIDATION
# ============================================================

total_completed_rows = int(
    production_df[
        "completed_state_rows"
    ].sum()
)

total_remaining_missing = int(
    production_df[
        "missing_state_after"
    ].sum()
)

total_topology_changes = int(
    production_df[
        "topology_changes"
    ].sum()
)

total_state_carried = int(
    production_df[
        "state_available_current_0"
    ].sum()
)

print("=" * 70)
print("GRAPH DATASET V1.0 VALIDATION")
print("=" * 70)

print(
    "Archives completed:",
    len(production_df)
)

print(
    "Archives failed:",
    len(failure_df)
)

print(
    "Rows causally completed:",
    total_completed_rows
)

print(
    "Rows marked carried-state:",
    total_state_carried
)

print(
    "Remaining missing current-state timestamps:",
    total_remaining_missing
)

print(
    "Current topology values changed:",
    total_topology_changes
)

expected_ok = (
    len(production_df) == EXPECTED_ARCHIVES
    and
    len(failure_df) == 0
    and
    total_completed_rows == EXPECTED_COMPLETED_ROWS
    and
    total_state_carried == EXPECTED_COMPLETED_ROWS
    and
    total_remaining_missing == 0
    and
    total_topology_changes == 0
)

status = (
    "PASS"
    if expected_ok
    else
    "REVIEW_REQUIRED"
)

final_report = pd.DataFrame({
    "metric": [
        "archives_completed",
        "archives_failed",
        "rows_causally_completed",
        "rows_marked_carried_state",
        "remaining_missing_state_rows",
        "topology_values_changed",
        "dataset_v1_status"
    ],
    "value": [
        len(production_df),
        len(failure_df),
        total_completed_rows,
        total_state_carried,
        total_remaining_missing,
        total_topology_changes,
        status
    ]
})

final_report.to_csv(
    OUTPUT_ROOT /
    "04_GRAPH_DATASET_V1_FINAL_REPORT.csv",
    index=False
)

display(final_report)

print("\nFinal status:", status)
print("V1.0 output:", OUTPUT_ROOT)


## Expected final result

We expect:

- Archives completed: **225**
- Archives failed: **0**
- Rows causally completed: **1,792**
- Rows marked `state_available_current = 0`: **1,792**
- Remaining missing current-state timestamps: **0**
- Current topology values changed: **0**
- Final status: **PASS**

If these conditions hold, the graph dataset can be frozen as **VeReMi Graph Dataset V1.0**.
